In [2]:
import pandas as pd

df = pd.read_csv("C:/Users/Trang Ha/Documents/Data_Game/Cookie_Cats.csv")
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,9.0,False,False
1,337,gate_30,48.0,True,True
2,377,gate_40,2.0,False,False
3,483,gate_40,6.0,True,False
4,488,gate_40,NaN,True,True


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   userid          90189 non-null  int64  
 1   version         90189 non-null  str    
 2   sum_gamerounds  78741 non-null  float64
 3   retention_1     90189 non-null  bool   
 4   retention_7     90189 non-null  bool   
dtypes: bool(2), float64(1), int64(1), str(1)
memory usage: 2.2 MB


In [3]:
df.isnull().sum()

userid                0
version               0
sum_gamerounds    11448
retention_1           0
retention_7           0
dtype: int64

In [4]:
df = df.drop_duplicates(subset=['userid'], keep='first')

In [5]:
df["retention_1"].value_counts()

retention_1
False    49120
True     41069
Name: count, dtype: int64

In [6]:
df["retention_1"] = df["retention_1"].astype(int)
df["retention_7"] = df["retention_7"].astype(int)

In [7]:
df.describe()

,userid,sum_gamerounds,retention_1,retention_7
count,9.018900e+04,78741.000000,90189.000000,90189.000000
mean,4.998412e+06,54.318347,0.455366,0.247624
std,2.883286e+06,83.385139,0.498007,0.431635
min,1.160000e+02,0.000000,0.000000,0.000000
25%,2.512230e+06,5.000000,0.000000,0.000000
50%,4.995815e+06,18.000000,0.000000,0.000000
75%,7.496452e+06,69.000000,1.000000,0.000000
max,9.999861e+06,1069.000000,1.000000,1.000000


In [9]:
df["sum_gamerounds_new"] = df.groupby(
    ["version", "retention_1", "retention_7"]
)["sum_gamerounds"].transform(lambda group: group.fillna(group.median()))
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7,sum_gamerounds_new
0,116,gate_30,9.0,0,0,9.0
1,337,gate_30,48.0,1,1,48.0
2,377,gate_40,2.0,0,0,2.0
3,483,gate_40,6.0,1,0,6.0
4,488,gate_40,NaN,1,1,18.0


In [12]:
median_by_group = (
    df.groupby(["version", "retention_1", "retention_7"])["sum_gamerounds"]
      .transform("median")
)

mask_median = df["sum_gamerounds"].isna()

df.loc[mask_median, "sum_gamerounds_new"] = median_by_group[mask_median]

mask_zero = (
    df["sum_gamerounds"].isna() &
    (df["retention_1"] == False) &
    (df["retention_7"] == False)
)
df.loc[mask_zero, "sum_gamerounds_new"] = 0
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7,sum_gamerounds_new
0,116,gate_30,9.0,0,0,9.0
1,337,gate_30,48.0,1,1,48.0
2,377,gate_40,2.0,0,0,2.0
3,483,gate_40,6.0,1,0,6.0
4,488,gate_40,NaN,1,1,18.0


In [13]:
df[
    (df["version"] == "gate_40") &
    (df["retention_1"] == 1) &
    (df["retention_7"] == 1) &
    (df["sum_gamerounds"].isna())
]

,userid,version,sum_gamerounds,retention_1,retention_7,sum_gamerounds_new
4,488,gate_40,NaN,1,1,18.0
466,48201,gate_40,NaN,1,1,18.0
533,55250,gate_40,NaN,1,1,18.0
863,96745,gate_40,NaN,1,1,18.0
1021,113087,gate_40,NaN,1,1,18.0
...,...,...,...,...,...,...
89588,9934927,gate_40,NaN,1,1,18.0
89732,9951192,gate_40,NaN,1,1,18.0
89767,9954781,gate_40,NaN,1,1,18.0
89768,9954937,gate_40,NaN,1,1,18.0


In [ ]:
df.drop(columns=["sum_gamerounds"], inplace=True)


KeyError: "['sum_gamerounds'] not found in axis"

In [19]:
df.rename(columns={"sum_gamerounds_new": "sum_gamerounds"}, inplace=True)
df.head()

,userid,version,retention_1,retention_7,sum_gamerounds
0,116,gate_30,0,0,9.0
1,337,gate_30,1,1,48.0
2,377,gate_40,0,0,2.0
3,483,gate_40,1,0,6.0
4,488,gate_40,1,1,18.0


In [20]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

connection_string = (
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=localhost\SQLEXPRESS;"
    "DATABASE=gaming;"
    "Trusted_Connection=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(connection_string)
)
df.to_sql("gaming_data",engine, if_exists="replace", index=False)

104